# HW3. From a Raw File to an Analysis-Ready Table: NYC Yellow Taxi Trips

**Author:** Serhii Kravchenko
**Dataset:** NYC TLC Yellow Taxi Trip Records, March 2024, plus the TLC Taxi Zone lookup
**Environment:** Google Colab (also runs locally on Windows, macOS and Linux), PostgreSQL 16 via `pgserver`, SQL cells via `jupysql`

The notebook takes the official monthly Parquet file all the way to a typed and audited table: sampling, raw import into a staging table with `COPY FROM STDIN`, a typed table filled with `INSERT INTO ... SELECT`, a data-quality audit, distributions and anomalies, ten EDA queries and a reflection. Every run starts with `DROP TABLE IF EXISTS ... CASCADE`, so **Restart session and run all** works from a clean state.

## 1. Setup and loading

### 1.1 Environment

`pgserver` ships PostgreSQL 16 as a Python package, so the database runs inside the notebook session.

A few environment details:
- Colab now defaults to Python 3.13, while `pgserver` 0.1.4 only publishes wheels up to Python 3.12. Its native module is built for the stable ABI (`abi3`), so the 3.12 wheel works on 3.13 once the wheel tag is renamed. The install cell handles this automatically.
- Taxi timestamps are New York wall-clock time without an offset. On Colab (Linux) PostgreSQL resolves `America/New_York`. The Windows build of `pgserver` comes without the time zone database, so there the notebook falls back to the POSIX rule `EST5EDT,M3.2.0,M11.1.0`: UTC−5, and UTC−4 from the second Sunday of March to the first Sunday of November. That is exactly the US rule in force since 2007, so both give the same result for 2024. The zone is detected once and stored in `NYC_TZ`, and SQL cells use it as `'{{NYC_TZ}}'`. The saved outputs come from a local Windows run, so they show the POSIX name.
- The session `TimeZone` is set to the same zone, so `TIMESTAMPTZ` values are displayed in New York time.
- `XDG_RUNTIME_DIR` is set only to silence a harmless `platformdirs` warning.

In [1]:
import glob, os, subprocess, sys, tempfile

os.environ.setdefault("XDG_RUNTIME_DIR", tempfile.mkdtemp())


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


pip_install("jupysql", "psycopg2-binary", "sqlalchemy", "pandas", "pyarrow")

try:
    import pgserver
except ImportError:
    if sys.version_info >= (3, 13):
        tmp = tempfile.mkdtemp()
        subprocess.check_call([
            sys.executable, "-m", "pip", "download", "pgserver==0.1.4",
            "--no-deps", "--only-binary=:all:", "--python-version", "3.12",
            "-d", tmp, "-q",
        ])
        wheel = glob.glob(os.path.join(tmp, "pgserver-*.whl"))[0]
        fixed = wheel.replace("-cp312-cp312-", "-cp312-abi3-")
        os.rename(wheel, fixed)
        pip_install(fixed)
    else:
        pip_install("pgserver==0.1.4")
    import pgserver

print("Python", sys.version.split()[0], "| pgserver ready")

Python 3.11.0 | pgserver ready


In [2]:
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import DBAPIError

PGDATA = Path("/content/hw3_pg") if Path("/content").exists() else Path.cwd() / "pgdata"
pg = pgserver.get_server(PGDATA, cleanup_mode="stop")
DB_URI = pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1)

probe = create_engine(DB_URI)
with probe.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

try:
    with probe.connect() as conn:
        conn.execute(text("SELECT TIMESTAMP '2024-03-10 12:00' AT TIME ZONE 'America/New_York'"))
    NYC_TZ = "America/New_York"
except DBAPIError:
    NYC_TZ = "EST5EDT,M3.2.0,M11.1.0"
probe.dispose()

print("NYC_TZ =", NYC_TZ)

PostgreSQL 16.2 on x86_64-pc-mingw64, compiled by gcc.exe (x86_64-posix-seh-rev2, Built by MinGW-W64 project) 12.2.0, 64-bit
NYC_TZ = EST5EDT,M3.2.0,M11.1.0


In [3]:
engine = create_engine(DB_URI, connect_args={"options": f"-c timezone={NYC_TZ}"})


def q(query: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)


%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 50
%sql engine

In [4]:
%%sql
SELECT current_setting('server_version') AS pg_version,
       current_setting('TimeZone')       AS session_tz;

pg_version,session_tz
16.2,"EST5EDT,M3.2.0,M11.1.0"


### 1.2 Getting the data

**Source:** [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), file *Yellow Taxi Trip Records, March 2024* (`yellow_tripdata_2024-03.parquet`, about 60 MB, 3.58 million trips), and the *Taxi Zone Lookup Table* from the same page. Field meanings follow the official [data dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf).

**Why March 2024.** The month includes the switch to daylight saving time (Sunday, March 10), which is a good test for the local-time to `TIMESTAMPTZ` conversion.

**Columns.** 15 of the 19 columns are kept: everything needed for trip time, place, distance, rate, payment and the main money fields. `extra`, `mta_tax`, `tolls_amount` and `improvement_surcharge` are dropped, because they are small fixed components of `total_amount` that no question in this notebook uses.

**Sample.** The file is ordered by time, so `head(100_000)` would only cover March 1. Instead, the notebook takes a seeded random sample of 100,000 **trips** over the whole month. The sampling unit is the trip key (`VendorID`, pickup time, dropoff time, pickup zone, dropoff zone), not the row, because TLC writes corrections as extra records with the same key (an original row plus a reversal with negated amounts). Sampling single rows would break those pairs apart and hide them from the duplicate audit, so all records of a sampled key are kept. With `random_state=42` the result is the same on every run.

The Parquet file is cached in the temp folder. The CSV goes to `data/`, which is also committed to the repository.

In [5]:
import shutil
from urllib.request import urlretrieve

DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet"
ZONES_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
SAMPLE_TRIPS = 100_000
RANDOM_STATE = 42

DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(exist_ok=True)
parquet_path = Path(tempfile.gettempdir()) / "yellow_tripdata_2024-03.parquet"
csv_path = DATA_DIR / "hw3_taxi_sample.csv"
zones_path = DATA_DIR / "taxi_zone_lookup.csv"

if not parquet_path.exists():
    partial, _ = urlretrieve(DATA_URL)
    shutil.move(partial, parquet_path)
urlretrieve(ZONES_URL, zones_path)

columns = [
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
    "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "tip_amount", "total_amount",
    "congestion_surcharge", "Airport_fee",
]
key_cols = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
            "PULocationID", "DOLocationID"]

full_df = pd.read_parquet(parquet_path, columns=columns)
trip_key = full_df.groupby(key_cols).ngroup()
sampled_keys = trip_key.drop_duplicates().sample(n=SAMPLE_TRIPS, random_state=RANDOM_STATE)

df = full_df[trip_key.isin(sampled_keys)]
df.to_csv(csv_path, index=False, lineterminator="\n")

print(f"Full month:  {len(full_df):,} rows, {trip_key.nunique():,} trip keys, "
      f"{len(full_df) - trip_key.nunique():,} rows repeat a key")
print(f"Full month:  {(full_df['tpep_pickup_datetime'] < '2009-01-01').sum()} pickups before 2009, "
      f"{(full_df['tpep_dropoff_datetime'] < full_df['tpep_pickup_datetime']).sum()} dropoffs before pickup")
print(f"Sample:      {len(df):,} rows for {SAMPLE_TRIPS:,} trip keys")
print(f"Pickups:     {df['tpep_pickup_datetime'].min()} .. {df['tpep_pickup_datetime'].max()}")
print(f"CSV:         {csv_path.name}, {csv_path.stat().st_size / 1e6:.1f} MB")
df.head(3)

Full month:  3,582,628 rows, 3,543,724 trip keys, 38,904 rows repeat a key
Full month:  2 pickups before 2009, 117 dropoffs before pickup
Sample:      101,102 rows for 100,000 trip keys
Pickups:     2024-02-29 23:37:21 .. 2024-03-31 23:59:46
CSV:         hw3_taxi_sample.csv, 8.9 MB


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,tip_amount,total_amount,congestion_surcharge,Airport_fee
29,2,2024-03-01 00:53:32,2024-03-01 01:08:18,1.0,4.05,1.0,N,142,116,1,19.8,4.96,29.76,2.5,0.0
104,1,2024-03-01 00:36:25,2024-03-01 01:05:52,1.0,10.70,1.0,N,249,21,1,41.5,9.30,55.80,2.5,0.0
175,2,2024-03-01 00:20:39,2024-03-01 00:31:02,1.0,1.31,1.0,N,100,170,1,10.7,5.00,20.70,2.5,0.0


### 1.3 Staging table

Every raw column is `TEXT`. A value like `1.0` in an integer column, an empty field or an unexpected marker can't stop the import, and the original text stays available for the audit. The columns follow the CSV order and get a `_raw` suffix.

In [6]:
%%sql
DROP TABLE IF EXISTS hw3_taxi_trips   CASCADE;
DROP TABLE IF EXISTS hw3_taxi_staging CASCADE;
DROP TABLE IF EXISTS hw3_taxi_zones   CASCADE;

CREATE TABLE hw3_taxi_staging (
    vendor_id_raw             TEXT,
    pickup_raw                TEXT,
    dropoff_raw               TEXT,
    passenger_count_raw       TEXT,
    trip_distance_raw         TEXT,
    ratecode_id_raw           TEXT,
    store_and_fwd_flag_raw    TEXT,
    pu_location_id_raw        TEXT,
    do_location_id_raw        TEXT,
    payment_type_raw          TEXT,
    fare_amount_raw           TEXT,
    tip_amount_raw            TEXT,
    total_amount_raw          TEXT,
    congestion_surcharge_raw  TEXT,
    airport_fee_raw           TEXT
);

SELECT COUNT(*) AS hw3_tables
FROM pg_tables
WHERE tablename LIKE 'hw3\_%';

hw3_tables
1


### 1.4 Loading with `COPY FROM STDIN`

`COPY` maps fields by position, so the CSV header is checked against the expected column order first. In CSV mode an unquoted empty field becomes `NULL` and a quoted `""` stays an empty string. The cleaning step handles both through `NULLIF(TRIM(...), '')`.

In [7]:
import csv

with open(csv_path, encoding="utf-8", newline="") as file:
    header = next(csv.reader(file))
assert header == columns, header


def copy_from_stdin(copy_sql: str, path: Path) -> None:
    raw_conn = engine.raw_connection()
    try:
        with raw_conn.cursor() as cur, open(path, "r", encoding="utf-8", newline="") as file:
            cur.copy_expert(copy_sql, file)
        raw_conn.commit()
    finally:
        raw_conn.close()


copy_from_stdin(
    """
    COPY hw3_taxi_staging (
        vendor_id_raw,
        pickup_raw,
        dropoff_raw,
        passenger_count_raw,
        trip_distance_raw,
        ratecode_id_raw,
        store_and_fwd_flag_raw,
        pu_location_id_raw,
        do_location_id_raw,
        payment_type_raw,
        fare_amount_raw,
        tip_amount_raw,
        total_amount_raw,
        congestion_surcharge_raw,
        airport_fee_raw
    )
    FROM STDIN
    WITH (
        FORMAT csv,
        HEADER true,
        DELIMITER ','
    )
    """,
    csv_path,
)

loaded = q("SELECT COUNT(*) AS loaded_rows FROM hw3_taxi_staging")
assert loaded["loaded_rows"][0] == len(df)
loaded

,loaded_rows
0,101102


In [8]:
%%sql
SELECT vendor_id_raw, pickup_raw, dropoff_raw, passenger_count_raw,
       trip_distance_raw, ratecode_id_raw, store_and_fwd_flag_raw,
       payment_type_raw, fare_amount_raw, airport_fee_raw
FROM hw3_taxi_staging
LIMIT 5;

vendor_id_raw,pickup_raw,dropoff_raw,passenger_count_raw,trip_distance_raw,ratecode_id_raw,store_and_fwd_flag_raw,payment_type_raw,fare_amount_raw,airport_fee_raw
2,2024-03-01 00:53:32,2024-03-01 01:08:18,1.0,4.05,1.0,N,1,19.8,0.0
1,2024-03-01 00:36:25,2024-03-01 01:05:52,1.0,10.7,1.0,N,1,41.5,0.0
2,2024-03-01 00:20:39,2024-03-01 00:31:02,1.0,1.31,1.0,N,1,10.7,0.0
2,2024-03-01 00:21:57,2024-03-01 00:30:00,1.0,3.11,1.0,N,1,14.9,0.0
1,2024-03-01 00:14:42,2024-03-01 00:25:20,1.0,1.6,1.0,N,1,11.4,0.0


### 1.5 Reference table: taxi zones

Trip rows only carry zone IDs. The official lookup maps each `LocationID` to a borough, a zone name and a service zone. It is a small reference file (265 rows of IDs and plain text), so it goes straight into a typed table: if any value didn't parse, `COPY` would fail, which is the right behaviour for a reference. IDs 264 (`Unknown`) and 265 (`Outside of NYC`) are TLC codes for an unknown or out-of-city location. They are kept as rows because trips reference them, and section 3 counts how often that happens.

In [9]:
%%sql
CREATE TABLE hw3_taxi_zones (
    location_id   SMALLINT  PRIMARY KEY
        CHECK (location_id > 0),
    borough       TEXT      NOT NULL,
    zone          TEXT      NOT NULL,
    service_zone  TEXT      NOT NULL
);

SELECT tablename
FROM pg_tables
WHERE tablename LIKE 'hw3\_%'
ORDER BY tablename;

tablename
hw3_taxi_staging
hw3_taxi_zones


In [10]:
copy_from_stdin(
    """
    COPY hw3_taxi_zones (location_id, borough, zone, service_zone)
    FROM STDIN
    WITH (FORMAT csv, HEADER true, DELIMITER ',')
    """,
    zones_path,
)

q("""
    SELECT location_id, borough, zone, service_zone
    FROM hw3_taxi_zones
    WHERE location_id IN (1, 132, 138, 264, 265)
       OR location_id = (SELECT MAX(location_id) FROM hw3_taxi_zones)
    ORDER BY location_id
""")

,location_id,borough,zone,service_zone
0,1,EWR,Newark Airport,EWR
1,132,Queens,JFK Airport,Airports
2,138,Queens,LaGuardia Airport,Airports
3,264,Unknown,N/A,N/A
4,265,N/A,Outside of NYC,N/A


## 2. Staging and DDL

### 2.1 Column decisions

| Clean column | Raw column | Meaning | Source format | Type | Cleaning rule |
|---|---|---|---|---|---|
| `trip_id` | — | technical surrogate key | — | `BIGINT` identity, `PRIMARY KEY` | generated; the source has no trip ID |
| `vendor_id` | `VendorID` | TPEP provider: 1 Creative Mobile Technologies, 2 Curb Mobility, 6 Myle, 7 Helix | integer as text | `SMALLINT NOT NULL`, `CHECK IN (1, 2, 6, 7)` | `TRIM`, empty → `NULL`, cast |
| `pickup_at`, `dropoff_at` | `tpep_pickup_datetime`, `tpep_dropoff_datetime` | moments the meter was engaged and disengaged | `YYYY-MM-DD HH:MI:SS`, New York wall-clock time, no offset | `TIMESTAMPTZ NOT NULL` | cast to `TIMESTAMP`, then `AT TIME ZONE '{{NYC_TZ}}'`. A trip is a moment in time, and durations must stay correct across the DST switch (see 2.4) |
| `pickup_date` | derived | New York calendar day of the pickup | — | `DATE NOT NULL`, generated column | `(pickup_at AT TIME ZONE '{{NYC_TZ}}')::DATE`; generated, so it can't drift from `pickup_at` |
| `passenger_count` | `passenger_count` | passengers, entered by the driver | `1.0`-style float text, because the column has gaps; empty for Flex Fare trips | `SMALLINT`, nullable, `CHECK BETWEEN 0 AND 9` | `::NUMERIC::SMALLINT`. `0` is not a documented code, so it is kept and flagged in section 4 |
| `trip_distance` | `trip_distance` | miles reported by the taximeter | decimal text, 2 decimals | `NUMERIC(8, 2) NOT NULL`, `CHECK (>= 0)` | exact decimal instead of float; 8 digits fit the largest value in the month (176,836.3, itself an anomaly) |
| `ratecode_id` | `RatecodeID` | final rate: 1 standard, 2 JFK, 3 Newark, 4 Nassau/Westchester, 5 negotiated, 6 group ride | float text; `99` = "Null/unknown" per the dictionary; empty for Flex Fare | `SMALLINT`, nullable, `CHECK IN (1, …, 6)` | `99` → `NULL` with `NULLIF`, because the dictionary defines it as missing |
| `is_store_and_forward` | `store_and_fwd_flag` | the record was held in vehicle memory before upload | `Y` / `N` / empty | `BOOLEAN`, nullable | `Y` → `TRUE`, `N` → `FALSE`, anything else → `NULL` |
| `pu_location_id`, `do_location_id` | `PULocationID`, `DOLocationID` | TLC zone where the meter was engaged / disengaged | integer as text | `SMALLINT NOT NULL`, `REFERENCES hw3_taxi_zones` | cast; 264/265 stay, because they are valid lookup rows |
| `payment_type` | `payment_type` | 0 Flex Fare, 1 credit card, 2 cash, 3 no charge, 4 dispute, 5 unknown, 6 voided | integer as text | `SMALLINT NOT NULL`, `CHECK BETWEEN 0 AND 6` | cast; labels are added in queries with `CASE` |
| `fare_amount`, `tip_amount`, `total_amount` | same names | meter fare; tip (filled automatically for card payments, cash tips are not recorded); total charged | decimal text, up to 2 decimals, can be negative | `NUMERIC(10, 2) NOT NULL` | money with exact cents. Negative values are kept: they are reversals, so there is deliberately no `CHECK (>= 0)` |
| `congestion_surcharge`, `airport_fee` | `congestion_surcharge`, `Airport_fee` | NY State congestion surcharge; fee for pickups at LaGuardia and JFK | decimal text; empty for Flex Fare | `NUMERIC(10, 2)`, nullable | empty → `NULL`, not `0`: "not reported" is not "zero dollars" |

There is no `VARCHAR(n)` in the trip table, because no field has a business length limit; the zone names use `TEXT` for the same reason.

The `CHECK` constraints describe the domain: valid vendor, rate and payment codes from the data dictionary, a passenger count a taxi can hold, a non-negative distance, dropoff not before pickup, and no pickups before 2009, when TLC trip records start (anything earlier is a meter clock error).

### 2.2 Typed table

In [11]:
%%sql
CREATE TABLE hw3_taxi_trips (
    trip_id               BIGINT
        GENERATED BY DEFAULT AS IDENTITY
        PRIMARY KEY,

    vendor_id             SMALLINT        NOT NULL
        CHECK (vendor_id IN (1, 2, 6, 7)),

    pickup_at             TIMESTAMPTZ     NOT NULL
        CHECK (pickup_at >= TIMESTAMPTZ '2009-01-01 00:00:00-05'),
    dropoff_at            TIMESTAMPTZ     NOT NULL,
    pickup_date           DATE            NOT NULL
        GENERATED ALWAYS AS ((pickup_at AT TIME ZONE '{{NYC_TZ}}')::DATE) STORED,

    passenger_count       SMALLINT
        CHECK (passenger_count BETWEEN 0 AND 9),
    trip_distance         NUMERIC(8, 2)   NOT NULL
        CHECK (trip_distance >= 0),
    ratecode_id           SMALLINT
        CHECK (ratecode_id IN (1, 2, 3, 4, 5, 6)),
    is_store_and_forward  BOOLEAN,

    pu_location_id        SMALLINT        NOT NULL
        REFERENCES hw3_taxi_zones (location_id),
    do_location_id        SMALLINT        NOT NULL
        REFERENCES hw3_taxi_zones (location_id),

    payment_type          SMALLINT        NOT NULL
        CHECK (payment_type BETWEEN 0 AND 6),

    fare_amount           NUMERIC(10, 2)  NOT NULL,
    tip_amount            NUMERIC(10, 2)  NOT NULL,
    total_amount          NUMERIC(10, 2)  NOT NULL,
    congestion_surcharge  NUMERIC(10, 2),
    airport_fee           NUMERIC(10, 2),

    CHECK (dropoff_at >= pickup_at)
);

SELECT column_name, data_type, numeric_precision, numeric_scale,
       is_nullable, is_generated
FROM information_schema.columns
WHERE table_name = 'hw3_taxi_trips'
ORDER BY ordinal_position;

column_name,data_type,numeric_precision,numeric_scale,is_nullable,is_generated
trip_id,bigint,64,0,NO,NEVER
vendor_id,smallint,16,0,NO,NEVER
pickup_at,timestamp with time zone,None,None,NO,NEVER
dropoff_at,timestamp with time zone,None,None,NO,NEVER
pickup_date,date,None,None,NO,ALWAYS
passenger_count,smallint,16,0,YES,NEVER
trip_distance,numeric,8,2,NO,NEVER
ratecode_id,smallint,16,0,YES,NEVER
is_store_and_forward,boolean,None,None,YES,NEVER
pu_location_id,smallint,16,0,NO,NEVER


### 2.3 Staging → typed table

`parsed` converts every raw value. The `WHERE` keeps only rows the typed table can accept: both timestamps present, pickup not before 2009, dropoff not before pickup, and a non-negative distance. Everything else, including negative money and zero passengers, is loaded and examined later instead of being silently dropped.

In [12]:
%%sql
WITH parsed AS (
    SELECT
        NULLIF(TRIM(vendor_id_raw), '')::SMALLINT
            AS vendor_id,

        NULLIF(TRIM(pickup_raw), '')::TIMESTAMP
            AT TIME ZONE '{{NYC_TZ}}' AS pickup_at,

        NULLIF(TRIM(dropoff_raw), '')::TIMESTAMP
            AT TIME ZONE '{{NYC_TZ}}' AS dropoff_at,

        NULLIF(TRIM(passenger_count_raw), '')::NUMERIC::SMALLINT
            AS passenger_count,

        NULLIF(TRIM(trip_distance_raw), '')::NUMERIC(8, 2)
            AS trip_distance,

        NULLIF(NULLIF(TRIM(ratecode_id_raw), '')::NUMERIC::SMALLINT, 99)
            AS ratecode_id,

        CASE UPPER(TRIM(store_and_fwd_flag_raw))
            WHEN 'Y' THEN TRUE
            WHEN 'N' THEN FALSE
        END AS is_store_and_forward,

        NULLIF(TRIM(pu_location_id_raw), '')::SMALLINT
            AS pu_location_id,

        NULLIF(TRIM(do_location_id_raw), '')::SMALLINT
            AS do_location_id,

        NULLIF(TRIM(payment_type_raw), '')::SMALLINT
            AS payment_type,

        NULLIF(TRIM(fare_amount_raw), '')::NUMERIC(10, 2)
            AS fare_amount,

        NULLIF(TRIM(tip_amount_raw), '')::NUMERIC(10, 2)
            AS tip_amount,

        NULLIF(TRIM(total_amount_raw), '')::NUMERIC(10, 2)
            AS total_amount,

        NULLIF(TRIM(congestion_surcharge_raw), '')::NUMERIC(10, 2)
            AS congestion_surcharge,

        NULLIF(TRIM(airport_fee_raw), '')::NUMERIC(10, 2)
            AS airport_fee
    FROM hw3_taxi_staging
)
INSERT INTO hw3_taxi_trips (
    vendor_id,
    pickup_at,
    dropoff_at,
    passenger_count,
    trip_distance,
    ratecode_id,
    is_store_and_forward,
    pu_location_id,
    do_location_id,
    payment_type,
    fare_amount,
    tip_amount,
    total_amount,
    congestion_surcharge,
    airport_fee
)
SELECT
    vendor_id,
    pickup_at,
    dropoff_at,
    passenger_count,
    trip_distance,
    ratecode_id,
    is_store_and_forward,
    pu_location_id,
    do_location_id,
    payment_type,
    fare_amount,
    tip_amount,
    total_amount,
    congestion_surcharge,
    airport_fee
FROM parsed
WHERE pickup_at IS NOT NULL
  AND dropoff_at IS NOT NULL
  AND pickup_at >= TIMESTAMPTZ '2009-01-01 00:00:00-05'
  AND dropoff_at >= pickup_at
  AND trip_distance >= 0;

SELECT COUNT(*) AS clean_rows
FROM hw3_taxi_trips;

clean_rows
101099


The constraints PostgreSQL actually created (`p` primary key, `f` foreign key, `c` check):

In [13]:
%%sql
SELECT conname                   AS constraint_name,
       contype                   AS type,
       pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'hw3_taxi_trips'::regclass
ORDER BY contype, conname;

constraint_name,type,definition
hw3_taxi_trips_check,c,CHECK ((dropoff_at >= pickup_at))
hw3_taxi_trips_passenger_count_check,c,CHECK (((passenger_count >= 0) AND (passenger_count <= 9)))
hw3_taxi_trips_payment_type_check,c,CHECK (((payment_type >= 0) AND (payment_type <= 6)))
hw3_taxi_trips_pickup_at_check,c,CHECK ((pickup_at >= '2009-01-01 00:00:00-05'::timestamp with time zone))
hw3_taxi_trips_ratecode_id_check,c,"CHECK ((ratecode_id = ANY (ARRAY[1, 2, 3, 4, 5, 6])))"
hw3_taxi_trips_trip_distance_check,c,CHECK ((trip_distance >= (0)::numeric))
hw3_taxi_trips_vendor_id_check,c,"CHECK ((vendor_id = ANY (ARRAY[1, 2, 6, 7])))"
hw3_taxi_trips_do_location_id_fkey,f,FOREIGN KEY (do_location_id) REFERENCES hw3_taxi_zones(location_id)
hw3_taxi_trips_pu_location_id_fkey,f,FOREIGN KEY (pu_location_id) REFERENCES hw3_taxi_zones(location_id)
hw3_taxi_trips_pkey,p,PRIMARY KEY (trip_id)


### 2.4 Why `TIMESTAMPTZ`: the DST switch

On Sunday, March 10, 2024, New York clocks jumped from 02:00 to 03:00. If the raw values really are local wall-clock time, there should be no pickups in the hour that didn't exist. The raw text can be checked directly with `LIKE`:

In [14]:
%%sql
SELECT LEFT(pickup_raw, 13) AS local_hour,
       COUNT(*)             AS n_pickups
FROM hw3_taxi_staging
WHERE pickup_raw LIKE '2024-03-10 0%'
GROUP BY local_hour
ORDER BY local_hour;

local_hour,n_pickups
2024-03-10 00,160
2024-03-10 01,160
2024-03-10 03,132
2024-03-10 04,74
2024-03-10 05,32
2024-03-10 06,24
2024-03-10 07,41
2024-03-10 08,57
2024-03-10 09,73


There are pickups at 00:xx, 01:xx and 03:xx, but none at 02:xx, because that hour didn't exist. So the raw values are New York wall-clock time with DST applied, and `AT TIME ZONE` with the New York rule is the correct conversion. A fixed −05:00 offset would be wrong for every trip after March 10.

For trips that span the switch (02:00 EST = 07:00 UTC), the difference of the wall-clock values is one hour too long, while `dropoff_at - pickup_at` on `TIMESTAMPTZ` gives the real duration:

In [15]:
%%sql
SELECT trip_id,
       pickup_at,
       dropoff_at,
       (dropoff_at AT TIME ZONE '{{NYC_TZ}}')
           - (pickup_at AT TIME ZONE '{{NYC_TZ}}')  AS wall_clock_difference,
       dropoff_at - pickup_at                       AS real_duration,
       trip_distance
FROM hw3_taxi_trips
WHERE pickup_at  >= TIMESTAMPTZ '2024-03-10 05:00:00+00'
  AND pickup_at  <  TIMESTAMPTZ '2024-03-10 07:00:00+00'
  AND dropoff_at >= TIMESTAMPTZ '2024-03-10 07:00:00+00'
ORDER BY pickup_at, trip_id;

trip_id,pickup_at,dropoff_at,wall_clock_difference,real_duration,trip_distance
26868,2024-03-10 01:23:40-05:00,2024-03-10 03:09:49-04:00,1:46:09,0:46:09,20.90
26890,2024-03-10 01:31:48-05:00,2024-03-10 03:15:58-04:00,1:44:10,0:44:10,27.07
26839,2024-03-10 01:40:06-05:00,2024-03-10 03:17:21-04:00,1:37:15,0:37:15,14.60
26879,2024-03-10 01:40:57-05:00,2024-03-10 03:10:34-04:00,1:29:37,0:29:37,5.80
92497,2024-03-10 01:46:29-05:00,2024-03-10 03:00:05-04:00,1:13:36,0:13:36,4.32
26891,2024-03-10 01:47:00-05:00,2024-03-10 03:18:17-04:00,1:31:17,0:31:17,26.02
26892,2024-03-10 01:47:00-05:00,2024-03-10 03:18:17-04:00,1:31:17,0:31:17,26.02
26855,2024-03-10 01:47:05-05:00,2024-03-10 03:02:07-04:00,1:15:02,0:15:02,4.50
92474,2024-03-10 01:47:32-05:00,2024-03-10 03:02:04-04:00,1:14:32,0:14:32,0.00
26920,2024-03-10 01:49:56-05:00,2024-03-10 03:16:35-04:00,1:26:39,0:26:39,14.66


Every trip in this list looks exactly one hour longer on the wall clock than it really was. For example, the trip from 01:51:12 to 03:00:35 took 9 minutes, not 1 h 9 min. Duration feeds speed, fare per minute and the outlier rules in 4.3, so storing `TIMESTAMPTZ` fixes it once, at load time, instead of in every query.

## 3. Data-quality audit

### 3.1 Row counts

In [16]:
%%sql
SELECT 'staging' AS layer, COUNT(*) AS n_rows
FROM hw3_taxi_staging

UNION ALL

SELECT 'clean' AS layer, COUNT(*) AS n_rows
FROM hw3_taxi_trips;

layer,n_rows
staging,101102
clean,101099


Which filter removed each excluded row (the first rule that fails, in the same order as the `WHERE` in 2.3):

In [17]:
%%sql
WITH parsed AS (
    SELECT
        NULLIF(TRIM(pickup_raw), '')::TIMESTAMP  AT TIME ZONE '{{NYC_TZ}}' AS pickup_at,
        NULLIF(TRIM(dropoff_raw), '')::TIMESTAMP AT TIME ZONE '{{NYC_TZ}}' AS dropoff_at,
        NULLIF(TRIM(trip_distance_raw), '')::NUMERIC                        AS trip_distance
    FROM hw3_taxi_staging
)
SELECT
    CASE
        WHEN pickup_at IS NULL OR dropoff_at IS NULL                THEN 'missing timestamp'
        WHEN pickup_at < TIMESTAMPTZ '2009-01-01 00:00:00-05'       THEN 'pickup before 2009'
        WHEN dropoff_at < pickup_at                                 THEN 'dropoff before pickup'
        WHEN trip_distance IS NULL OR trip_distance < 0             THEN 'missing or negative distance'
        ELSE 'kept'
    END      AS outcome,
    COUNT(*) AS n_rows
FROM parsed
GROUP BY outcome
ORDER BY n_rows DESC;

outcome,n_rows
kept,101099
dropoff before pickup,3


Staging holds 101,102 rows and the typed table 101,099. All 3 missing rows were removed by a single rule, `dropoff_at >= pickup_at`: their dropoff is recorded before the pickup. That is a meter clock error, and there is no way to tell which of the two timestamps is wrong. The other rules didn't fire in this sample, but the full month needs them: it contains 2 pickups from 2002 and 117 dropoffs before pickup (output in 1.2). Rows that are questionable but still possible, such as negative amounts, zero distance or zero passengers, are loaded on purpose and examined in section 4.

### 3.2 Duplicates

There is no trip ID in the source, so the business key is the combination that identifies a physical trip: **vendor + pickup moment + dropoff moment + pickup zone + dropoff zone**. One meter can't start two different trips in the same second with the same end time and zones. This is still a heuristic, not a guaranteed natural key, so the check finds exact or likely duplicates only.

In [18]:
%%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id))
        AS distinct_business_keys,
    COUNT(*) - COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id))
        AS duplicate_rows
FROM hw3_taxi_trips;

total_rows,distinct_business_keys,duplicate_rows
101099,99997,1102


What the repeated keys look like:

In [19]:
%%sql
WITH key_groups AS (
    SELECT vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id,
           COUNT(*)                     AS n_rows,
           SUM(total_amount)            AS net_total,
           COUNT(*) FILTER (WHERE total_amount < 0) AS n_negative,
           STRING_AGG(DISTINCT payment_type::TEXT, ',') AS payment_types
    FROM hw3_taxi_trips
    GROUP BY vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id
    HAVING COUNT(*) > 1
)
SELECT n_rows,
       n_negative,
       net_total = 0     AS nets_to_zero,
       payment_types,
       COUNT(*)          AS n_keys
FROM key_groups
GROUP BY n_rows, n_negative, nets_to_zero, payment_types
ORDER BY n_keys DESC;

n_rows,n_negative,nets_to_zero,payment_types,n_keys
2,1,True,4,682
2,1,True,2,221
2,1,True,3,181
2,1,False,4,9
2,1,False,2,6
2,1,False,3,2
2,1,True,1,1


In [20]:
%%sql
SELECT trip_id, vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id,
       payment_type, fare_amount, tip_amount, total_amount
FROM hw3_taxi_trips
WHERE (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id) IN (
    SELECT vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id
    FROM hw3_taxi_trips
    GROUP BY vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id
    HAVING COUNT(*) > 1
)
ORDER BY pickup_at, trip_id
LIMIT 6;

trip_id,vendor_id,pickup_at,dropoff_at,pu_location_id,do_location_id,payment_type,fare_amount,tip_amount,total_amount
32,2,2024-03-01 00:23:51-05:00,2024-03-01 00:29:57-05:00,79,234,2,-8.60,0.00,-13.60
33,2,2024-03-01 00:23:51-05:00,2024-03-01 00:29:57-05:00,79,234,2,8.60,0.00,13.60
70,2,2024-03-01 00:29:57-05:00,2024-03-01 00:30:12-05:00,231,231,4,-3.00,0.00,-8.00
71,2,2024-03-01 00:29:57-05:00,2024-03-01 00:30:12-05:00,231,231,4,3.00,0.00,8.00
117,2,2024-03-01 01:18:59-05:00,2024-03-01 01:21:22-05:00,229,140,4,-5.10,0.00,-10.10
118,2,2024-03-01 01:18:59-05:00,2024-03-01 01:21:22-05:00,229,140,4,5.10,0.00,10.10


There are 1,102 repeated business keys, 1.1% of the rows. That is the same share as in the full month (38,904 out of 3.58 million), which shows the key-level sampling kept these pairs intact. Every repeated key has exactly two rows, and in each pair exactly one row has a negative total: an original trip plus its reversal. Both rows carry the final payment status, mostly 4 (dispute), then 2 (cash) and 3 (no charge). 1,085 pairs net to exactly zero, and 17 are partial reversals.

These are accounting corrections, not technical duplicates to delete. For revenue, both rows must stay so the net is right. For a fare or demand model, the pair is one cancelled or refunded trip, and both rows should be left out. The key is a heuristic, so two genuinely different trips with identical second-precision timestamps and zones would also be counted here, but every pair found follows the reversal pattern.

### 3.3 NULL rates and special markers

NULL share for key columns of the typed table:

In [21]:
%%sql
SELECT
    COUNT(*) AS total_rows,

    ROUND(COUNT(*) FILTER (WHERE passenger_count IS NULL) * 100.0
          / NULLIF(COUNT(*), 0), 2) AS pct_null_passenger_count,

    ROUND(COUNT(*) FILTER (WHERE ratecode_id IS NULL) * 100.0
          / NULLIF(COUNT(*), 0), 2) AS pct_null_ratecode_id,

    ROUND(COUNT(*) FILTER (WHERE is_store_and_forward IS NULL) * 100.0
          / NULLIF(COUNT(*), 0), 2) AS pct_null_store_fwd,

    ROUND(COUNT(*) FILTER (WHERE congestion_surcharge IS NULL) * 100.0
          / NULLIF(COUNT(*), 0), 2) AS pct_null_congestion,

    ROUND(COUNT(*) FILTER (WHERE airport_fee IS NULL) * 100.0
          / NULLIF(COUNT(*), 0), 2) AS pct_null_airport_fee,

    ROUND(COUNT(*) FILTER (WHERE fare_amount IS NULL) * 100.0
          / NULLIF(COUNT(*), 0), 2) AS pct_null_fare_amount,

    ROUND(COUNT(*) FILTER (WHERE pu_location_id IS NULL) * 100.0
          / NULLIF(COUNT(*), 0), 2) AS pct_null_pu_location
FROM hw3_taxi_trips;

total_rows,pct_null_passenger_count,pct_null_ratecode_id,pct_null_store_fwd,pct_null_congestion,pct_null_airport_fee,pct_null_fare_amount,pct_null_pu_location
101099,11.89,12.95,11.89,11.89,11.89,0.00,0.00


The raw layer, scanned column by column for empty values and the usual text markers (`?`, `unknown`, `NA`, `N/A`, `nan`, `null`, `none`). `to_jsonb` turns each staging row into key/value pairs, so one query covers all 15 columns:

In [22]:
%%sql
SELECT
    kv.key AS raw_column,
    COUNT(*) FILTER (WHERE kv.value IS NULL OR TRIM(kv.value) = '') AS n_empty,
    COUNT(*) FILTER (
        WHERE LOWER(TRIM(kv.value)) IN ('?', 'unknown', 'na', 'n/a', 'nan', 'null', 'none')
    ) AS n_text_marker
FROM hw3_taxi_staging AS s
CROSS JOIN LATERAL jsonb_each_text(to_jsonb(s)) AS kv
GROUP BY kv.key
ORDER BY n_empty DESC, kv.key;

raw_column,n_empty,n_text_marker
airport_fee_raw,12023,0
congestion_surcharge_raw,12023,0
passenger_count_raw,12023,0
ratecode_id_raw,12023,0
store_and_fwd_flag_raw,12023,0
do_location_id_raw,0,0
dropoff_raw,0,0
fare_amount_raw,0,0
payment_type_raw,0,0
pickup_raw,0,0


Numeric codes that mean "unknown" are not caught by a text scan, so they are counted separately, per vendor (`ROLLUP` adds the total row):

In [23]:
%%sql
SELECT
    COALESCE(vendor_id_raw, 'all vendors') AS vendor,
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE TRIM(ratecode_id_raw) IN ('99', '99.0'))
        AS ratecode_99_unknown,
    COUNT(*) FILTER (WHERE TRIM(passenger_count_raw) IN ('0', '0.0'))
        AS zero_passengers,
    COUNT(*) FILTER (WHERE TRIM(trip_distance_raw) IN ('0', '0.0'))
        AS zero_distance,
    COUNT(*) FILTER (WHERE TRIM(pu_location_id_raw) IN ('264', '265')
                        OR TRIM(do_location_id_raw) IN ('264', '265'))
        AS zone_264_or_265,
    COUNT(*) FILTER (WHERE TRIM(payment_type_raw) = '5')
        AS payment_unknown
FROM hw3_taxi_staging
GROUP BY ROLLUP (vendor_id_raw)
ORDER BY vendor_id_raw NULLS LAST;

vendor,total_rows,ratecode_99_unknown,zero_passengers,zero_distance,zone_264_or_265,payment_unknown
1,24599,1069,1144,1650,463,0
2,76491,0,6,859,544,0
6,12,0,0,0,12,0
all vendors,101102,1069,1150,2509,1019,0


Five raw columns are empty in exactly the same 12,023 staging rows (11.89% after cleaning): `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge` and `Airport_fee`. This is a structural block, not random noise, and Q4 shows that it is exactly the Flex Fare trips. `ratecode_id` has a higher NULL rate (12.95%) because the 1,069 raw `99` values ("Null/unknown") were turned into NULL, and every one of them comes from vendor 1. The text scan found no `?`, `unknown` or `N/A` markers, and the key fields (timestamps, zones, payment type, amounts) have no gaps at all.

A zero NULL rate doesn't mean the data is complete, though. Missing information also hides behind valid-looking codes: 1,150 trips with 0 passengers (1,144 of them from vendor 1), 2,509 with zero distance, and 1,019 that touch zone 264 (Unknown) or 265 (Outside of NYC), including all 12 rows from vendor 6. None of this shows up in a NULL count, which is why sentinel values are checked separately, and why several of them turn out to be vendor-specific.

## 4. Distributions and anomalies

### 4.1 Payment type

In [24]:
%%sql
SELECT
    payment_type,
    CASE payment_type
        WHEN 0 THEN 'Flex Fare'
        WHEN 1 THEN 'Credit card'
        WHEN 2 THEN 'Cash'
        WHEN 3 THEN 'No charge'
        WHEN 4 THEN 'Dispute'
        WHEN 5 THEN 'Unknown'
        WHEN 6 THEN 'Voided trip'
    END AS payment_label,
    COUNT(*) AS n_rows,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_rows
FROM hw3_taxi_trips
GROUP BY payment_type
ORDER BY n_rows DESC, payment_type
LIMIT 10;

payment_type,payment_label,n_rows,pct_rows
1,Credit card,73481,72.68
2,Cash,13178,13.03
0,Flex Fare,12020,11.89
4,Dispute,1736,1.72
3,No charge,684,0.68


Credit cards dominate with 72.7% of rows, followed by cash (13.0%) and Flex Fare (11.9%). Disputes (1.7%) and no-charge trips (0.7%) are rare, but they contain most of the reversal rows from 3.2. Codes 5 (unknown) and 6 (voided) don't occur at all. As a feature this is an imbalanced categorical variable. More importantly, it is only known after the trip, and it decides whether a tip is recorded at all (Q6).

### 4.2 Numeric columns: fare and distance

The template statistics plus the median and the 99th percentile, which show how far the mean is pulled by the tail:

In [25]:
%%sql
SELECT
    'fare_amount'                                            AS metric,
    COUNT(fare_amount)                                       AS n_non_null,
    MIN(fare_amount)                                         AS min_value,
    MAX(fare_amount)                                         AS max_value,
    ROUND(AVG(fare_amount)::NUMERIC, 2)                      AS avg_value,
    ROUND(STDDEV_SAMP(fare_amount)::NUMERIC, 2)              AS sd_value,
    PERCENTILE_CONT(0.5)  WITHIN GROUP (ORDER BY fare_amount) AS median_value,
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY fare_amount) AS p99_value
FROM hw3_taxi_trips

UNION ALL

SELECT
    'trip_distance',
    COUNT(trip_distance),
    MIN(trip_distance),
    MAX(trip_distance),
    ROUND(AVG(trip_distance)::NUMERIC, 2),
    ROUND(STDDEV_SAMP(trip_distance)::NUMERIC, 2),
    PERCENTILE_CONT(0.5)  WITHIN GROUP (ORDER BY trip_distance),
    PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY trip_distance)
FROM hw3_taxi_trips;

metric,n_non_null,min_value,max_value,avg_value,sd_value,median_value,p99_value
fare_amount,101099,-800.00,800.00,18.63,18.40,13.5,75.1
trip_distance,101099,0.00,51657.73,4.25,220.55,1.74,19.95


The fare is right-skewed: the median is $13.50, the mean $18.63, and 99% of fares are below $75.10. The ±$800 extremes are a single negotiated trip and its reversal. Distance is far more distorted. The median is 1.74 miles and p99 about 20 miles, but the maximum of 51,657.73 miles inflates the standard deviation to 220.55, more than 100 times the median. Mean + 3σ is about 666 miles, so a 3σ rule would catch the two absurd readings but miss a 151-mile trip that took 11 minutes. Medians, percentiles and domain rules describe this data better.

### 4.3 Domain outlier check

A 3σ rule doesn't work well here: the distributions are heavily skewed, and a few broken odometer readings inflate the standard deviation so much that the threshold ends up above most of the errors. Domain rules are clearer. Each one is a fact about a NYC taxi trip:

In [26]:
%%sql
WITH t AS (
    SELECT trip_id, pickup_date, trip_distance, fare_amount, tip_amount, total_amount,
           passenger_count, EXTRACT(EPOCH FROM dropoff_at - pickup_at) / 60 AS duration_min
    FROM hw3_taxi_trips
)
SELECT
    COUNT(*) FILTER (WHERE fare_amount < 0 OR tip_amount < 0 OR total_amount < 0)
        AS negative_amounts,
    COUNT(*) FILTER (WHERE trip_distance = 0)
        AS zero_distance,
    COUNT(*) FILTER (WHERE duration_min < 1)
        AS under_1_minute,
    COUNT(*) FILTER (WHERE duration_min > 6 * 60)
        AS over_6_hours,
    COUNT(*) FILTER (WHERE trip_distance > 100)
        AS over_100_miles,
    COUNT(*) FILTER (WHERE duration_min > 0 AND trip_distance / (duration_min / 60) > 80)
        AS faster_than_80_mph,
    COUNT(*) FILTER (WHERE total_amount > 500)
        AS total_over_500,
    COUNT(*) FILTER (WHERE passenger_count = 0)
        AS zero_passengers,
    COUNT(*) FILTER (WHERE pickup_date NOT BETWEEN DATE '2024-03-01' AND DATE '2024-03-31')
        AS pickup_outside_march,
    COUNT(*) AS total_rows
FROM t;

negative_amounts,zero_distance,under_1_minute,over_6_hours,over_100_miles,faster_than_80_mph,total_over_500,zero_passengers,pickup_outside_march,total_rows
1695,2509,1187,50,3,38,1,1150,2,101099


Where the negative amounts come from: whether the row has a twin with the same business key (3.2), and whether the total is still positive:

In [27]:
%%sql
WITH key_counts AS (
    SELECT vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id,
           COUNT(*) AS n_rows
    FROM hw3_taxi_trips
    GROUP BY vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id
)
SELECT t.payment_type,
       k.n_rows > 1                                   AS has_twin_row,
       COUNT(*)                                       AS n_negative_rows,
       COUNT(*) FILTER (WHERE t.total_amount >= 0)    AS n_total_not_negative
FROM hw3_taxi_trips AS t
JOIN key_counts AS k
  USING (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id)
WHERE t.fare_amount < 0 OR t.tip_amount < 0 OR t.total_amount < 0
GROUP BY t.payment_type, has_twin_row
ORDER BY t.payment_type, has_twin_row;

payment_type,has_twin_row,n_negative_rows,n_total_not_negative
0,False,429,429
1,True,1,0
2,False,32,0
2,True,227,0
3,False,26,0
3,True,183,0
4,False,106,0
4,True,691,0


The physically impossible rows (implied speed above 80 mph, over 100 miles, over 6 hours, or a total above $500), largest distance first:

In [28]:
%%sql
SELECT trip_id, vendor_id, pickup_at, dropoff_at,
       ROUND(EXTRACT(EPOCH FROM dropoff_at - pickup_at) / 60, 1) AS duration_min,
       trip_distance,
       ROUND(trip_distance / NULLIF(EXTRACT(EPOCH FROM dropoff_at - pickup_at) / 3600, 0), 1)
           AS implied_mph,
       ratecode_id, payment_type, fare_amount, total_amount
FROM hw3_taxi_trips
WHERE trip_distance > 100
   OR dropoff_at - pickup_at > INTERVAL '6 hours'
   OR total_amount > 500
   OR (dropoff_at > pickup_at
       AND trip_distance / (EXTRACT(EPOCH FROM dropoff_at - pickup_at) / 3600) > 80)
ORDER BY trip_distance DESC, trip_id
LIMIT 20;

trip_id,vendor_id,pickup_at,dropoff_at,duration_min,trip_distance,implied_mph,ratecode_id,payment_type,fare_amount,total_amount
94489,2,2024-03-16 01:53:00-04:00,2024-03-16 01:59:00-04:00,6.0,51657.73,516577.3,None,0,9.40,13.40
93556,2,2024-03-14 07:45:00-04:00,2024-03-14 08:20:00-04:00,35.0,47413.57,81280.4,None,0,34.54,38.54
37966,2,2024-03-14 07:09:42-04:00,2024-03-14 07:20:39-04:00,11.0,150.96,827.2,1,1,14.20,18.84
71242,2,2024-03-25 16:00:44-04:00,2024-03-26 00:00:00-04:00,479.3,26.21,3.3,2,1,70.00,114.00
95405,6,2024-03-18 06:03:36-04:00,2024-03-18 06:03:49-04:00,0.2,21.80,6036.9,None,0,63.20,64.00
49321,2,2024-03-17 18:48:40-04:00,2024-03-17 18:51:35-04:00,2.9,20.05,412.5,1,1,84.90,110.19
60079,2,2024-03-21 16:02:07-04:00,2024-03-22 00:00:00-04:00,477.9,18.09,2.3,2,2,70.00,82.69
81838,2,2024-03-29 11:49:24-04:00,2024-03-30 10:58:59-04:00,1389.6,17.54,0.8,2,1,70.00,89.03
22487,1,2024-03-08 18:10:37-05:00,2024-03-08 18:11:01-05:00,0.4,17.40,2610.0,5,4,0.00,1.00
85309,2,2024-03-30 15:41:34-04:00,2024-03-31 15:22:58-04:00,1421.4,17.39,0.7,3,1,90.90,107.28


| Rule | Rows | What it is | Action |
|---|---|---|---|
| Negative amounts | 1,695 | 1,102 are halves of the reversal pairs from 3.2. 429 are Flex Fare rows with a small negative `fare_amount` but a non-negative total, which looks like a balancing entry rather than a refund. The other 164 are reversals whose original isn't in the file under the same key | keep, add an `is_adjustment` flag; exclude from fare and tip training; keep for revenue |
| Zero distance | 2,509 | the meter recorded no movement: cancelled pickups, flat or negotiated prices, broken odometers | flag; exclude from per-mile and speed features; keep for demand counts |
| Under 1 minute | 1,187 | false starts and meters closed right away | flag; exclude from duration and fare models |
| Over 6 hours | 50 | the meter wasn't switched off: some trips end at exactly 00:00:00 or last almost 24 hours with an ordinary distance and fare | keep the fare; treat the duration as missing for features |
| Over 100 miles / faster than 80 mph | 3 / 38 | odometer or clock errors: 51,657.73 miles in 6 minutes, 150.96 miles in 11 minutes, 21.8 miles in 13 seconds | treat the distance as missing for training; send the trips to the vendor for review |
| Total over $500 | 1 | the negotiated $800 trip with zero distance | possible but unverifiable: keep with a flag and check with the source |
| Zero passengers | 1,150 | 0 isn't a possible count for a paid trip; almost all come from vendor 1 (3.3) | treat as missing in features, keep the raw value; ask the vendor to make the field mandatory |
| Pickup outside March | 2 | trips that started on Feb 29 shortly before midnight | valid: keep, and filter by `pickup_date` for monthly aggregates |

Nothing is removed just because it is large. The only rows dropped in 2.3 broke a hard physical rule (dropoff before pickup). Everything else stays in the table, and the decision is made per use case: revenue reports need the reversals, a fare model needs them excluded.

## 5. Ten DQL/EDA queries

Times are grouped in New York local time (`AT TIME ZONE '{{NYC_TZ}}'`). Most queries exclude reversal rows (negative amounts) when the question is about real trips.

### Q1. Pickups by hour of day

**Business-question:** At which hours of the day (New York time) do pickups peak, and how do distance and fare change with the hour?

Reversals and the three broken odometer readings from 4.3 are left out; two of those readings alone would add 18–26 miles to the average distance of their hour.

In [29]:
%%sql
SELECT
    EXTRACT(HOUR FROM pickup_at AT TIME ZONE '{{NYC_TZ}}')::INT  AS pickup_hour,
    COUNT(*)                                                    AS n_trips,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)          AS pct_trips,
    ROUND(AVG(trip_distance), 2)                                AS avg_distance_mi,
    ROUND(AVG(fare_amount), 2)                                  AS avg_fare
FROM hw3_taxi_trips
WHERE fare_amount >= 0
  AND trip_distance <= 100
GROUP BY pickup_hour
ORDER BY pickup_hour;

pickup_hour,n_trips,pct_trips,avg_distance_mi,avg_fare
0,3006,3.02,3.62,19.55
1,1963,1.97,3.30,18.07
2,1223,1.23,2.98,17.10
3,921,0.93,3.20,17.24
4,663,0.67,4.08,21.47
5,609,0.61,5.31,25.72
6,1377,1.39,4.70,23.12
7,2619,2.63,3.59,19.56
8,3833,3.86,3.06,18.54
9,3945,3.97,2.97,18.26


**Interpretation:** Pickups peak in the evening: 18:00 is the busiest hour (7,192 trips, 7.2%), and the eight hours from 14:00 to 21:59 account for about half of all trips. The morning commute is much flatter (about 3,900 trips per hour at 8–9 am), and the quietest hours are 4–5 am with about 0.6% each. Those early trips are the longest and most expensive on average (5.31 miles and $25.72 at 5 am), which fits early airport runs. Hour of day is clearly a strong feature for demand forecasting, and it has to be computed in local time.

### Q2. Busiest pickup zones

**Business-question:** Which ten pickup zones generate the most trips, and what share of all trips do they cover?

In [30]:
%%sql
SELECT
    z.zone                                               AS pickup_zone,
    z.borough,
    COUNT(*)                                             AS n_trips,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)   AS pct_trips
FROM hw3_taxi_trips AS t
JOIN hw3_taxi_zones AS z
  ON z.location_id = t.pu_location_id
WHERE t.total_amount >= 0
GROUP BY z.zone, z.borough
ORDER BY n_trips DESC, pickup_zone
LIMIT 10;

pickup_zone,borough,n_trips,pct_trips
Midtown Center,Manhattan,4573,4.58
Upper East Side South,Manhattan,4338,4.35
JFK Airport,Queens,4284,4.29
Upper East Side North,Manhattan,4121,4.13
Times Sq/Theatre District,Manhattan,3440,3.45
Midtown East,Manhattan,3418,3.42
Penn Station/Madison Sq West,Manhattan,3295,3.30
Lincoln Square East,Manhattan,3199,3.20
LaGuardia Airport,Queens,2962,2.97
Midtown North,Manhattan,2900,2.90


**Interpretation:** Midtown Center is the busiest pickup zone (4,573 trips, 4.6%), followed by Upper East Side South and JFK Airport. Eight of the top ten zones are in Manhattan, and the other two are JFK and LaGuardia. Together they cover about 37% of all trips, while 226 zones have at least one pickup (Q8). The location feature is therefore highly concentrated: a model sees thousands of examples for Midtown and only a handful for most outer-borough zones.

### Q3. Airport pickups

**Business-question:** How do pickups at the three airports compare with the rest of the city by volume, distance and amount charged?

In [31]:
%%sql
SELECT
    CASE
        WHEN z.zone ILIKE '%airport%' THEN z.zone
        ELSE 'Not an airport'
    END                                                  AS pickup_area,
    COUNT(*)                                             AS n_trips,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)   AS pct_trips,
    ROUND(AVG(t.trip_distance), 2)                       AS avg_distance_mi,
    ROUND(AVG(t.total_amount), 2)                        AS avg_total_amount
FROM hw3_taxi_trips AS t
JOIN hw3_taxi_zones AS z
  ON z.location_id = t.pu_location_id
WHERE t.total_amount > 0
GROUP BY pickup_area
ORDER BY n_trips DESC;

pickup_area,n_trips,pct_trips,avg_distance_mi,avg_total_amount
Not an airport,92556,92.73,3.55,23.93
JFK Airport,4284,4.29,15.92,82.66
LaGuardia Airport,2962,2.97,9.65,67.37
Newark Airport,14,0.01,0.89,90.71


**Interpretation:** JFK and LaGuardia together account for 7.3% of pickups, but they are a different kind of trip: on average 15.92 and 9.65 miles and $82.66 and $67.37 charged, against 3.55 miles and $23.93 everywhere else. Newark has only 14 pickups, and they look wrong: an average distance of 0.89 miles but $90.71 charged. Yellow cabs rarely pick up in New Jersey, so these are most likely trips *to* Newark with an incorrect pickup zone. An airport flag is worth adding as a feature, and the Newark pickups should go back to the source for checking.

### Q4. Where passenger_count is missing

**Business-question:** Is the missing `passenger_count` spread randomly, or is it concentrated in one payment type, and does the rest of the NULL block go missing with it?

In [32]:
%%sql
SELECT
    payment_type,
    COUNT(*)                                                      AS n_trips,
    COUNT(*) FILTER (WHERE passenger_count IS NULL)               AS n_passenger_null,
    ROUND(COUNT(*) FILTER (WHERE passenger_count IS NULL) * 100.0
          / COUNT(*), 2)                                          AS pct_passenger_null,
    COUNT(*) FILTER (
        WHERE passenger_count IS NULL
          AND is_store_and_forward IS NULL
          AND congestion_surcharge IS NULL
          AND airport_fee IS NULL
    )                                                             AS n_whole_block_null
FROM hw3_taxi_trips
GROUP BY payment_type
ORDER BY payment_type;

payment_type,n_trips,n_passenger_null,pct_passenger_null,n_whole_block_null
0,12020,12020,100.00,12020
1,73481,0,0.00,0
2,13178,0,0.00,0
3,684,0,0.00,0
4,1736,0,0.00,0


**Interpretation:** Missing `passenger_count` is not random at all: 100% of Flex Fare trips (payment type 0, 12,020 rows) have it NULL, and no other payment type has a single NULL. In each of those rows the whole block goes missing together: passenger count, store-and-forward flag, congestion surcharge and airport fee. This is a structural gap in how Flex Fare records are produced, so imputing a global mean would be wrong. A model should use a Flex Fare indicator or handle that segment separately.

### Q5. Weekday rush hour vs. other times

**Business-question:** How much slower are weekday rush-hour trips (7–9 am and 4–6 pm, Monday to Friday) than trips at other times?

Speed is total miles divided by total hours, so very short trips can't dominate it. One `NOT (...)` removes disputes, no-charge trips and the broken records from 4.3.

In [33]:
%%sql
SELECT
    CASE
        WHEN EXTRACT(ISODOW FROM pickup_date) BETWEEN 1 AND 5
         AND (   EXTRACT(HOUR FROM pickup_at AT TIME ZONE '{{NYC_TZ}}') BETWEEN 7 AND 9
              OR EXTRACT(HOUR FROM pickup_at AT TIME ZONE '{{NYC_TZ}}') BETWEEN 16 AND 18)
        THEN 'weekday rush hour'
        ELSE 'other time'
    END                                                                     AS time_bucket,
    COUNT(*)                                                                AS n_trips,
    ROUND(AVG(trip_distance), 2)                                            AS avg_distance_mi,
    ROUND(AVG(EXTRACT(EPOCH FROM dropoff_at - pickup_at) / 60), 1)          AS avg_duration_min,
    ROUND(SUM(trip_distance)
          / NULLIF(SUM(EXTRACT(EPOCH FROM dropoff_at - pickup_at)) / 3600, 0), 1)
                                                                            AS avg_speed_mph
FROM hw3_taxi_trips
WHERE trip_distance > 0
  AND fare_amount > 0
  AND NOT (   payment_type IN (3, 4)
           OR trip_distance > 100
           OR dropoff_at - pickup_at > INTERVAL '6 hours')
GROUP BY time_bucket
ORDER BY time_bucket DESC;

time_bucket,n_trips,avg_distance_mi,avg_duration_min,avg_speed_mph
weekday rush hour,21892,3.09,16.9,11.0
other time,73870,3.43,15.9,13.0


**Interpretation:** Weekday rush-hour trips average 11.0 mph against 13.0 mph at other times, about 15% slower. They are slightly shorter (3.09 vs. 3.43 miles) and still take a minute longer (16.9 vs. 15.9 minutes). For an ETA or fare model, what matters is the combination of weekday and hour, not the hour on its own.

### Q6. Tips by payment type

**Business-question:** How often and how much do riders tip when paying by card versus cash, as recorded in the data?

`NULLIF(fare_amount, 0)` turns zero fares into `NULL`, so they drop out of the percentage instead of causing a division by zero.

In [34]:
%%sql
SELECT
    payment_type,
    CASE payment_type WHEN 1 THEN 'Credit card' WHEN 2 THEN 'Cash' END   AS payment_label,
    COUNT(*)                                                            AS n_trips,
    ROUND(COUNT(*) FILTER (WHERE tip_amount > 0) * 100.0 / COUNT(*), 1) AS pct_with_tip,
    ROUND(AVG(tip_amount), 2)                                           AS avg_tip,
    ROUND(AVG(tip_amount / NULLIF(fare_amount, 0)) * 100, 1)            AS avg_tip_pct_of_fare
FROM hw3_taxi_trips
WHERE payment_type IN (1, 2)
  AND fare_amount >= 0
GROUP BY payment_type
ORDER BY payment_type;

payment_type,payment_label,n_trips,pct_with_tip,avg_tip,avg_tip_pct_of_fare
1,Credit card,73480,94.5,4.27,25.1
2,Cash,12921,0.0,0.00,0.0


**Interpretation:** 94.5% of card trips include a tip, averaging $4.27 or 25.1% of the fare. Cash trips show a tip in 0.0% of rows. That doesn't mean cash riders never tip: as the data dictionary says, cash tips are simply not recorded. So `tip_amount` is missing-not-at-random for cash trips: a tip model has to be trained on card trips only, and an average tip over all trips understates real tipping.

### Q7. Airport fee consistency

**Business-question:** Do pickups at JFK and LaGuardia (zones 132 and 138) carry the $1.75 airport fee, and do any other pickups carry it by mistake?

The expected fee is $1.75 for those two zones and $0 elsewhere. `IS DISTINCT FROM` counts a `NULL` fee as a mismatch, while `<>` returns `NULL` for it and silently skips the row. Both counts are shown to make the difference visible.

In [35]:
%%sql
SELECT
    pickup_type,
    COUNT(*)                                                    AS n_trips,
    COUNT(*) FILTER (WHERE airport_fee IS DISTINCT FROM expected_fee)
                                                                AS n_mismatch_distinct_from,
    COUNT(*) FILTER (WHERE airport_fee <> expected_fee)         AS n_mismatch_not_equal,
    COUNT(*) FILTER (WHERE airport_fee IS NULL)                 AS n_fee_null
FROM (
    SELECT
        CASE WHEN pu_location_id IN (132, 138)
             THEN 'JFK / LaGuardia pickup' ELSE 'other pickup' END   AS pickup_type,
        CASE WHEN pu_location_id IN (132, 138)
             THEN 1.75 ELSE 0.00 END                                 AS expected_fee,
        airport_fee
    FROM hw3_taxi_trips
    WHERE total_amount >= 0
) AS t
GROUP BY pickup_type
ORDER BY pickup_type;

pickup_type,n_trips,n_mismatch_distinct_from,n_mismatch_not_equal,n_fee_null
JFK / LaGuardia pickup,7246,196,165,31
other pickup,92587,12382,393,11989


**Interpretation:** At JFK and LaGuardia, 7,050 of 7,246 pickups carry the $1.75 fee. `IS DISTINCT FROM` finds 196 mismatches: 165 with another value and 31 with a NULL fee, and `<>` misses those 31. Among other pickups, `<>` finds 393 trips charged a fee they shouldn't have, while `IS DISTINCT FROM` reports 12,382, because it also counts the 11,989 Flex Fare rows where no fee was reported. So the operator depends on the question: use `<>` for "a wrong fee was charged" and `IS DISTINCT FROM` for "the fee is not as expected, including not reported".

### Q8. Cardinality of the location features

**Business-question:** How many distinct pickup zones, dropoff zones and origin–destination pairs does the sample contain, i.e. how high-cardinality are the location features?

In [36]:
%%sql
SELECT
    COUNT(*)                                                  AS n_od_pairs,
    COUNT(*) FILTER (WHERE pu_location_id = do_location_id)   AS n_same_zone_pairs,
    COUNT(DISTINCT pu_location_id)                            AS n_pickup_zones,
    COUNT(DISTINCT do_location_id)                            AS n_dropoff_zones
FROM (
    SELECT DISTINCT pu_location_id, do_location_id
    FROM hw3_taxi_trips
) AS od_pairs;

n_od_pairs,n_same_zone_pairs,n_pickup_zones,n_dropoff_zones
7499,142,226,250


**Interpretation:** The sample covers 226 pickup zones, 250 dropoff zones and 7,499 origin–destination pairs, 142 of which start and end in the same zone. That is about 13 trips per pair on average, but the distribution is very skewed (Q2, Q10), so most pairs have only a few examples. One-hot encoding the pair would create thousands of sparse columns. Target encoding with smoothing, embeddings, or zone-level features (borough, airport flag) are better choices.

### Q9. Fares by rate code

**Business-question:** How do distance and fare differ across rate codes, including trips whose rate code is unknown?

In [37]:
%%sql
SELECT
    COALESCE(
        CASE ratecode_id
            WHEN 1 THEN 'Standard'
            WHEN 2 THEN 'JFK'
            WHEN 3 THEN 'Newark'
            WHEN 4 THEN 'Nassau or Westchester'
            WHEN 5 THEN 'Negotiated'
            WHEN 6 THEN 'Group ride'
        END,
        'Unknown (NULL)'
    )                                    AS rate_name,
    COUNT(*)                             AS n_trips,
    ROUND(AVG(trip_distance), 2)         AS avg_distance_mi,
    ROUND(AVG(fare_amount), 2)           AS avg_fare,
    MIN(fare_amount)                     AS min_fare,
    MAX(fare_amount)                     AS max_fare
FROM hw3_taxi_trips
WHERE fare_amount >= 0
  AND trip_distance <= 100
GROUP BY rate_name
ORDER BY n_trips DESC;

rate_name,n_trips,avg_distance_mi,avg_fare,min_fare,max_fare
Standard,82441,2.63,16.28,0.00,229.80
Unknown (NULL),12658,3.21,20.34,0.00,192.35
JFK,3264,17.42,69.89,0.00,70.00
Negotiated,592,4.75,68.88,0.00,800.00
Newark,266,16.00,84.50,0.00,157.40
Nassau or Westchester,183,21.25,114.12,3.00,450.30


**Interpretation:** Standard-rate trips make up 83% of the rows, at 2.63 miles and $16.28 on average. The flat airport rate is clearly visible: JFK trips average $69.89 with a maximum of exactly $70.00, the JFK flat fare, so for them distance barely affects the fare. Negotiated fares are the noisiest group, with 4.75 miles on average but fares up to $800. Trips with an unknown rate code (12,658: Flex Fare plus the `99` code) look like ordinary city trips (3.21 miles, $20.34), so they are better kept as a separate category than dropped.

### Q10. Top routes by revenue

**Business-question:** Which ten pickup–dropoff zone pairs bring in the most revenue? Revenue is the net `total_amount`, so reversals are included and subtract their refunds.

In [38]:
%%sql
SELECT
    pz.zone                         AS pickup_zone,
    dz.zone                         AS dropoff_zone,
    COUNT(*)                        AS n_trips,
    SUM(t.total_amount)             AS revenue,
    ROUND(AVG(t.total_amount), 2)   AS avg_total_amount
FROM hw3_taxi_trips AS t
JOIN hw3_taxi_zones AS pz ON pz.location_id = t.pu_location_id
JOIN hw3_taxi_zones AS dz ON dz.location_id = t.do_location_id
GROUP BY pz.zone, dz.zone
ORDER BY revenue DESC, pickup_zone, dropoff_zone
LIMIT 10;

pickup_zone,dropoff_zone,n_trips,revenue,avg_total_amount
JFK Airport,Outside of NYC,151,17292.99,114.52
JFK Airport,Times Sq/Theatre District,185,16834.60,91.00
LaGuardia Airport,Times Sq/Theatre District,207,15715.46,75.92
JFK Airport,Midtown South,129,12336.34,95.63
JFK Airport,Clinton East,119,10390.85,87.32
Upper East Side South,Upper East Side North,624,9564.96,15.33
JFK Airport,Murray Hill,99,9312.32,94.06
Times Sq/Theatre District,LaGuardia Airport,122,9220.22,75.58
Upper East Side North,Upper East Side South,536,8333.91,15.55
JFK Airport,Midtown North,92,8150.27,88.59


**Interpretation:** Airports dominate revenue: eight of the top ten pairs start or end at JFK or LaGuardia, and JFK → Outside of NYC comes first ($17,292.99 from 151 trips, $114.52 per trip). The Upper East Side South ↔ North pairs follow the opposite pattern: by far the most trips in this list (624 and 536), at only about $15 each, so they make the top ten through volume. For fleet planning, airport queues bring revenue per trip, while the Upper East Side brings steady short-trip demand.

### Coverage of the required constructs

| Construct | Queries |
|---|---|
| explicit column list in `SELECT` | all |
| `WHERE` with `AND` / `OR` / `NOT` and parentheses | Q5 |
| `IN` / `BETWEEN` | Q5 (`BETWEEN`, `IN`), Q6 (`IN`), Q7 (`IN`) |
| `LIKE` / `ILIKE` | Q3 (`ILIKE`), plus `LIKE` on raw text in 2.4 |
| `IS NULL` / `IS NOT NULL` | Q4 |
| `COALESCE` / `NULLIF` | Q9 (`COALESCE`), Q5 and Q6 (`NULLIF`) |
| `IS DISTINCT FROM` | Q7 |
| `DISTINCT` | Q8 (`SELECT DISTINCT`, `COUNT(DISTINCT ...)`) |
| `ORDER BY` with `LIMIT` | Q2, Q10 |
| `GROUP BY` | Q1, Q2, Q3, Q4, Q5, Q6, Q7, Q9, Q10 |

## 6. Reflection

**Data quality.** Explicit gaps are rare, but missing information is hidden elsewhere. Five columns are NULL together for every Flex Fare trip, `RatecodeID = 99` means "unknown", and 0 passengers or zones 264/265 often mean "not entered". About 1% of rows are reversals that repeat the key of the original trip. Some values are physically impossible: 51,657 miles in six minutes, meters left running for almost a day. Several patterns are vendor-specific (99 codes and zero passengers come from vendor 1, unknown zones from vendor 6).

**Features.** For a fare or duration model at pickup I would use the local pickup hour and weekday, `pickup_date` (for holidays), pickup and dropoff zones with borough and an airport flag, rate code, vendor and passenger count (0 treated as missing). I would drop `trip_id` and `is_store_and_forward` as technical fields, and train without reversal, dispute and Flex Fare rows.

**Leakage.** `dropoff_at` and duration, the metered `trip_distance`, `payment_type`, `tip_amount`, `total_amount` and the surcharges only exist after the trip ends; at pickup, only a route-based distance estimate is available. For a tip model, `payment_type` and the total are leakage too.

**Pipeline.** I'd ask the data engineer for UTC timestamps or an explicit offset, a stable trip ID with a link from each reversal to its original, real NULLs instead of 99 and 0, documented Flex Fare fields, and a distance-versus-duration check at ingestion.

**ML task.** The most natural fit is regression: fare or duration predicted at pickup. Aggregated by zone and hour, the same data supports demand forecasting.

## 7. Final checks

In [39]:
staging_rows = q("SELECT COUNT(*) AS n FROM hw3_taxi_staging")["n"][0]
clean_rows = q("SELECT COUNT(*) AS n FROM hw3_taxi_trips")["n"][0]
column_types = q("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'hw3_taxi_trips'
""").set_index("column_name")["data_type"]
constraint_types = q("""
    SELECT contype, COUNT(*) AS n
    FROM pg_constraint
    WHERE conrelid = 'hw3_taxi_trips'::regclass
    GROUP BY contype
""").set_index("contype")["n"]

assert q("SELECT current_setting('server_version_num')::INT AS v")["v"][0] >= 160000
assert staging_rows == len(df)
assert 0 <= staging_rows - clean_rows < staging_rows * 0.001
assert constraint_types["p"] == 1 and constraint_types["f"] == 2 and constraint_types["c"] >= 2
assert column_types["pickup_at"] == "timestamp with time zone"
assert column_types["pickup_date"] == "date"
assert column_types["fare_amount"] == "numeric"
assert column_types["is_store_and_forward"] == "boolean"
assert q("SELECT COUNT(*) AS n FROM hw3_taxi_trips WHERE ratecode_id = 99")["n"][0] == 0
assert q("""
    SELECT COUNT(*) AS n FROM hw3_taxi_trips
    WHERE pickup_date <> (pickup_at AT TIME ZONE '{tz}')::DATE
""".format(tz=NYC_TZ))["n"][0] == 0

print(f"staging rows: {staging_rows:,} | clean rows: {clean_rows:,}")
print("All checks passed")

staging rows: 101,102 | clean rows: 101,099
All checks passed
